# CNMFe pipeline — part 2: extract neuronal components

Picks up where **`01_load_and_motion_correct.ipynb`** left off: the motion-
corrected zarr (`mc.zarr`) is on disk. This notebook runs the rest of the
CNMFe pipeline:

1. **Load** the motion-corrected movie into RAM. *Why RAM?* Spatial /
   temporal updates need pixel-major random access. After the Phase A–B
   refactors the only full copy in RAM is the movie itself —
   `BackgroundSubtractor` and `compute_W` allocate only per-batch buffers
   on top.
2. **CORR / PNR diagnostic** — visualise where the neurons are *before*
   running extraction. Lets you pick `sigma`, `min_corr`, `min_pnr` from
   the data instead of guessing.
3. **Configure** `CNMFeParams` and run `CNMFe.fit(...)`. Internally:
   greedy CORR/PNR init (on a strided sample — see `init_stride`) →
   ring-model background → spatial LASSO → temporal OASIS deconvolution
   → merge duplicates, iterated `n_iter_main` times → auto-evaluation
   (sigma-aware footprint size + SNR amplitude filter).
4. **Inspect results** — footprints overlaid on the mean image, a few
   traces (`C`, `C + YrA` via `model.C_projected`, `S`).
5. **Save** with `model.save(output_dir)`. Reload anywhere with
   `CNMFe.load(output_dir)`.

The file layout `save()` writes matches `full_pipeline.py`, so downstream
analysis scripts work either way.

**For very long recordings (60k+ frames)** the in-RAM load above doesn't
scale. **Section 9** at the bottom of this notebook shows the
true T-streaming alternative: transpose `mc.zarr` to pixel-major chunks
once and run `fit(..., Y_flat_zarr=...)` so the full `(T, H, W)` array is
never materialised. Peak RAM stays bounded by `K·T·4` regardless of T.

## 1. Imports & paths

Point `MC_ZARR` at the `mc.zarr` produced by part 1 (or any motion-corrected
zarr — the extraction step doesn't care how it was produced).

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

from cnmfe.io import open_zarr
from cnmfe.pipeline import CNMFe, CNMFeParams
from cnmfe.preprocess import correlation_pnr

# Path to the motion-corrected zarr produced by part 1.
PROJECT_ROOT = Path('D:/code/claude_cnmfe')
MC_ZARR  = PROJECT_ROOT / 'demo_movies' / 'demo_session' / 'mc_output' / 'mc.zarr'
OUTPUT_DIR = MC_ZARR.parent / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'mc zarr     : {MC_ZARR}')
print(f'results dir : {OUTPUT_DIR}')

## 2. Load the corrected movie into RAM

The extraction steps need pixel-major access to every frame, so the movie
is materialised once as `(T, H, W)` float32. Peak working RAM during
extraction is roughly **just this array plus small per-batch buffers** —
`BackgroundSubtractor` is lazy (slices of `(I-W)·(Y-b0)` on demand) and
`compute_W` caches the ring weight matrix across BCD iterations, so the
per-step overhead is bounded by `K·T·4` bytes plus a small pixel-batch
slice.

**Rule of thumb:** the dominant cost is the loaded movie itself,
`T * H * W * 4` bytes. A 10k-frame 600×600 movie is ~14 GB. 60k frames at
the same dims is ~86 GB — switch to the **streaming workflow in
Section 9** rather than subsampling.

In [ ]:
mc_zarr = open_zarr(MC_ZARR)
T_full, H, W = mc_zarr.shape
ram_gb = T_full * H * W * 4 / 1e9

print(f'mc.zarr shape : {mc_zarr.shape}')
print(f'        chunks: {mc_zarr.chunks}')
print(f'        dtype : {mc_zarr.dtype}')
print(f'estimated RAM for full load (float32): {ram_gb:.1f} GB')

In [ ]:
# --- Optional time subsampling of the materialised movie ---------------------
# Controls the `(T, H, W)` float32 array we load into RAM. Extraction never
# holds more than one copy of this plus small per-batch buffers (Phase A+B
# refactors: BackgroundSubtractor is lazy, compute_W avoids the full
# residual). So at 600x600 the rule of thumb is T * H * W * 4 bytes — 14 GB
# at 10k frames, 86 GB at 60k.
#
# If your movie won't fit in RAM:
#  - Moderate (slight RAM pressure):  set TIME_STRIDE > 1 here. This
#    subsamples *everything* downstream (footprints, traces, mean image).
#  - Huge (60k+ frames): skip this cell and use the streaming workflow in
#    Section 9 below. `fit(..., Y_flat_zarr=...)` runs against an on-disk
#    pixel-major zarr; peak RAM is bounded by K·T·4 regardless of T.
#
# A more targeted knob lives on CNMFeParams: `init_stride`. That subsamples
# *only* the greedy CORR/PNR init step (which transiently allocates two
# more copies of the movie), leaving the rest of the pipeline at full T.
# `init_stride=None` (default) auto-selects max(1, T // 5000). Tune it in
# the params cell below if you want to be explicit.
TIME_STRIDE = 1

t0 = time.time()
if TIME_STRIDE > 1:
    movie = np.asarray(mc_zarr[::TIME_STRIDE], dtype=np.float32)
else:
    movie = np.asarray(mc_zarr, dtype=np.float32)
T = movie.shape[0]
print(f'loaded movie : shape={movie.shape}  dtype={movie.dtype}')
print(f'              ~{movie.nbytes / 1e9:.2f} GB in RAM   (load took {time.time()-t0:.1f}s)')

## 3. CORR / PNR diagnostic — find where the neurons are

Two summary images tell you where neurons live and let you set extraction
thresholds from data rather than guessing:

- **CORR** — at each pixel, the mean Pearson correlation with its 8
  neighbours over time. A neuron spans several pixels that all flash
  together → CORR is high inside the neuron.
- **PNR** (peak-to-noise ratio) — max ΔF / noise std at each pixel. Bright
  transients → high PNR.

Their product `CORR × PNR` is the seed score used by greedy init:
noisy-looking pixels with chance correlation get filtered out (low PNR),
bright artefacts get filtered out (low CORR).

A **center-surround PSF** (DoG-like, sums to zero) is applied before computing
CORR/PNR — this suppresses the diffuse 1p background and isolates structures
at the neuron scale defined by `sigma`. Set `center_psf=False` for 2p data.

**Pick `sigma`** ≈ the neuron *radius* in pixels. Eyeball a few neurons in
the mean image first.

In [ ]:
# Sigma in pixels — the neuron radius. Adjust to match your data.
SIGMA = 3.0

# correlation_pnr already strides internally if the movie is long; we just pass
# the full (possibly subsampled) movie.
t0 = time.time()
cn, pnr = correlation_pnr(movie, sigma=SIGMA, center_psf=True, n_jobs=-1)
print(f'CORR/PNR computed in {time.time()-t0:.1f}s')
print(f'  CORR range: [{cn.min():.3f}, {cn.max():.3f}]')
print(f'  PNR  range: [{pnr.min():.3f}, {pnr.max():.3f}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
im0 = axes[0].imshow(cn, cmap='inferno', vmin=0, vmax=1)
axes[0].set_title('CORR (local correlation)')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(pnr, cmap='inferno')
axes[1].set_title('PNR (peak / noise)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(cn * pnr, cmap='inferno')
axes[2].set_title('CORR x PNR (seed score)')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

# Pixel scatter to help pick thresholds.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(cn.ravel(), pnr.ravel(), s=0.5, alpha=0.25, c='steelblue')
ax.set_xlabel('CORR')
ax.set_ylabel('PNR')
ax.set_title(f'Pixel CORR vs PNR  (sigma = {SIGMA})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Pick `min_corr` and `min_pnr` to sit just below the visible cluster of bright pixels.')

## 4. Configure extraction parameters

All algorithm parameters live on `CNMFeParams`. The most-tuned ones for new
data:

| Param | Meaning | Typical |
|---|---|---|
| `sigma` | Neuron radius in pixels | 2–5 |
| `min_corr` | Min CORR for a seed | 0.6–0.9 |
| `min_pnr` | Min PNR for a seed | 5–15 |
| `max_neurons` | Stop greedy init after N (None = no cap) | None |
| `init_stride` | Greedy-init temporal stride; `None` auto = `max(1, T // 5000)` | None |
| `n_iter_main` | Spatial + temporal + merge cycles | 1–3 |
| `n_iter_temporal` | BCD passes inside each temporal update | 2 |
| `merge_thr_corr` | Min temporal r to merge two components | 0.85 |
| `merge_thr_overlap` | Min Jaccard footprint overlap to merge | 0.5 |
| `merge_centre_dist_factor` | Centre-distance fallback (× sigma) | 2.0 |
| `spatial_max_thr` | Footprint cleanup threshold (fraction of peak) | 0.1 |
| `min_pixel` | Auto-eval hard pixel-count floor per component | 3 |
| `auto_eval_snr_amp_thr` | Auto-eval mean-amplitude SNR threshold; `0` to disable | 3.0 |
| `global_ar` | `True` = one g for all neurons; `False` = per-neuron | `True` |
| `n_jobs` | CPU workers; `-1` = all cores | `-1` |

Auto-evaluation runs between the BCD loop and the final temporal pass —
see `cnmfe.evaluate.auto_evaluate_components`. A component must pass two
checks: a hard pixel-count floor (`min_pixel`) and a scale-invariant
mean-amplitude SNR check (`mean(a²) / mean(sn²) >= auto_eval_snr_amp_thr`)
that catches ghost components whose footprint sits at the pixel-noise
floor. Set `auto_eval_snr_amp_thr=0` to disable the SNR check.

Start with the values below — they're sane defaults for 1p miniscope data —
then refine after looking at extracted footprints and traces.

In [ ]:
params = CNMFeParams(
    # Detection
    sigma=SIGMA,
    min_corr=0.8,
    min_pnr=10.0,
    max_neurons=None,

    # Main refinement loop
    n_iter_main=2,
    n_iter_temporal=2,

    # Merging
    merge_thr_corr=0.85,
    merge_thr_overlap=0.5,
    merge_centre_dist_factor=2.0,

    # Spatial cleanup
    spatial_max_thr=0.1,

    # Temporal
    ar_order=1,
    global_ar=True,
    skip_first_deconv=True,        # NNLS on the first pass, OASIS afterwards

    # Greedy init: stride for the (T, H, W) sample passed to greedy_corr_pnr.
    # None auto-selects max(1, T // 5000). Footprints A are spatial, so
    # spatial recovery is unaffected; full-T traces are re-projected after init.
    init_stride=None,

    # Auto-evaluation (post-BCD ghost rejection): components must have at
    # least `min_pixel` non-zero pixels AND a mean-amplitude SNR above
    # `auto_eval_snr_amp_thr`. Set the SNR threshold to 0 to disable.
    min_pixel=3,
    auto_eval_snr_amp_thr=3.0,

    # Parallelism
    n_jobs=-1,
    device='cpu',                  # 'cuda' if you have CuPy + a GPU
)

# Resolve the auto stride for the print summary.
_init_stride_resolved = params.init_stride or max(1, T // 5000)

print('Extraction params:')
print(f'  sigma={params.sigma}  min_corr={params.min_corr}  min_pnr={params.min_pnr}')
print(f'  n_iter_main={params.n_iter_main}  n_iter_temporal={params.n_iter_temporal}')
print(f'  init_stride={_init_stride_resolved}  (T_init={T // _init_stride_resolved} of {T} frames)')
print(f'  auto-eval: min_pixel={params.min_pixel}  snr_amp_thr={params.auto_eval_snr_amp_thr}')
print(f'  global_ar={params.global_ar}  n_jobs={params.n_jobs}  device={params.device!r}')

## 5. Run extraction

`do_motion_correction=False` because the movie is already corrected. The
pipeline prints progress for each phase: greedy init, ring background,
spatial/temporal refinement, merging.

Expected wall time scales roughly with `T * H * W * n_iter_main`. On a 10k-
frame 600×600 movie with `n_jobs=-1` on 8 cores: a few minutes.

In [ ]:
model = CNMFe(params)

t0 = time.time()
model.fit(movie, do_motion_correction=False)
fit_elapsed = time.time() - t0

K = model.A.shape[1]
print(f'\nExtracted {K} components in {fit_elapsed:.1f}s')
print(f'  A    : {model.A.shape}  sparse footprints  (H*W = {H*W} pixels, K = {K})')
print(f'  C    : {model.C.shape}  OASIS-deconvolved traces')
print(f'  YrA  : {model.YrA.shape}  residual; C + YrA = noisy projected trace')
print(f'  S    : {model.S.shape}  spike trains')
print(f'  g    : per-component AR coefficients (first = {model.g[0][0]:.3f})')

## 6. Inspect results

Three quick visuals:

1. **Footprint contours** on the temporal mean image — every detected
   neuron should sit on a bright blob in the mean image.
2. **Footprint max-projection** alongside the CORR×PNR map for reference.
3. **Sample traces** for the brightest neurons: `C` (deconvolved, clean
   AR(1) shape) overlaid with `C + YrA` (shape-faithful noisy projection)
   and the spike train `S` underneath.

Bright blobs without a contour = missed detections (lower `min_corr` or
`min_pnr`). Contours on featureless background = false positives (raise them).

In [ ]:
mean_img = movie.mean(axis=0)
A_dense = np.asarray(model.A.todense()).reshape(H, W, K)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(mean_img, cmap='gray',
               vmin=np.percentile(mean_img, 1),
               vmax=np.percentile(mean_img, 99))
for k in range(K):
    fp = A_dense[..., k]
    if fp.max() <= 0:
        continue
    axes[0].contour(fp, levels=[fp.max() * 0.3], colors='lime', linewidths=0.7)
axes[0].set_title(f'Mean image + {K} footprint contours')
axes[0].axis('off')

axes[1].imshow(A_dense.max(axis=2), cmap='hot')
axes[1].set_title('Footprint max-projection')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Pick the N brightest neurons (by C peak) and plot them.
N_TRACES = min(6, K)

if K == 0:
    print('No components extracted — nothing to plot.')
else:
    peak_per_k = model.C.max(axis=1)
    top_k = np.argsort(-peak_per_k)[:N_TRACES]
    C_proj = model.C_projected   # = model.C + model.YrA — shape-faithful

    fig, axes = plt.subplots(N_TRACES, 1, figsize=(11, 1.6 * N_TRACES), sharex=True)
    if N_TRACES == 1:
        axes = [axes]
    for ax, k in zip(axes, top_k):
        ax.plot(C_proj[k], lw=0.7, color='steelblue', alpha=0.7, label='C + YrA')
        ax.plot(model.C[k], lw=1.0, color='forestgreen', label='C (OASIS)')
        ax2 = ax.twinx()
        ax2.bar(np.arange(T), model.S[k], width=1.0, color='tomato',
                alpha=0.6, linewidth=0, label='S')
        ax2.set_yticks([])
        ax.set_ylabel(f'k = {k}', fontsize=8)
        ax.legend(loc='upper right', fontsize=7)
    axes[-1].set_xlabel('Frame')
    plt.suptitle('Top neurons — C (green), C + YrA (blue), S (red bars)', y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Save results to disk

`model.save(output_dir)` writes the full result set as standalone files
(same layout `full_pipeline.py` uses, so existing analysis scripts keep
working). Reload anywhere with `CNMFe.load(output_dir)`.

Files written:

- `A.npz` — sparse CSC footprints `(H*W, K)`
- `C.npy` — OASIS-deconvolved traces `(K, T)`
- `S.npy` — spike trains `(K, T)`
- `YrA.npy` — residual traces; `C + YrA` (== `model.C_projected`) = noisy projected trace
- `C_raw.npy` — raw init traces
- `sn.npy` — per-pixel noise std `(H, W)`
- `b0.npy`, `W.npz` — ring-background baseline + weights
- `g.npy`, `sn_per_k.npy` — per-component AR coefs + noise std
- `params.json` — the `CNMFeParams` used
- `manifest.json` — `dims`, `K`, `T` (non-parameter metadata)
- `shifts.npy` — copied from `mc_output/` if present
- `run_info.json` — notebook-level metadata (mc_zarr path, time_stride, wall time)

In [ ]:
# model.save() writes A.npz, C.npy, S.npy, YrA.npy, sn.npy, b0.npy, W.npz,
# g.npy, sn_per_k.npy, params.json, and manifest.json.
model.save(OUTPUT_DIR)

# Carry MC shifts forward from part 1 if present; this notebook ran with
# do_motion_correction=False so model.shifts is None.
mc_shifts_src = MC_ZARR.parent / 'shifts.npy'
if mc_shifts_src.exists():
    import shutil
    shutil.copy(mc_shifts_src, OUTPUT_DIR / 'shifts.npy')

# Optional run-level metadata that isn't part of CNMFeParams.
run_info = {
    'mc_zarr': str(MC_ZARR),
    'movie_shape_loaded': list(movie.shape),
    'time_stride': TIME_STRIDE,
    'K_extracted': K,
    'wall_time_s': round(fit_elapsed, 2),
}
(OUTPUT_DIR / 'run_info.json').write_text(json.dumps(run_info, indent=2))

print(f'Results saved to {OUTPUT_DIR}/')
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.name:18s}  {p.stat().st_size / 1e6:7.2f} MB')

print('\nReload anywhere with:')
print(f"  from cnmfe.pipeline import CNMFe")
print(f"  model = CNMFe.load(r'{OUTPUT_DIR}')")
print(f"  C_proj = model.C_projected   # = model.C + model.YrA")

## 8. What's next

Reload the model in any downstream analysis script:

```python
from cnmfe.pipeline import CNMFe

model = CNMFe.load('results/')        # restores A, C, S, YrA, sn, b0, W, g, ...
A   = model.A                          # (H*W, K) sparse footprints
C   = model.C                          # (K, T)   OASIS-deconvolved
S   = model.S                          # (K, T)   spike trains
C_proj = model.C_projected             # (K, T)   = C + YrA, shape-faithful
params = model.params                  # CNMFeParams used for the fit
```

Use `model.C` (clean AR(1)) for spike-event detection and amplitude features;
use `model.C_projected` whenever the OASIS shape constraint would distort the
answer (cross-correlation with behaviour, regression against an external
signal, raw-fluorescence plotting).

If results look off:

- **Too few neurons** — lower `min_corr` and/or `min_pnr`. Check the CORR×PNR
  map: are bright spots being skipped?
- **Too many duplicates** — raise `merge_thr_corr`, lower
  `merge_centre_dist_factor`, or check that `sigma` matches the actual neuron
  size (too small `sigma` ⇒ multiple seeds per neuron).
- **Smeared / vasculature-shaped footprints** — raise `spatial_max_thr` so
  weak pixels are clipped.
- **Noisy / shrinking traces over iterations** — the AR cache should prevent
  this, but you can also try `global_ar=False` (per-neuron g) or lower
  `n_iter_main`.


## 9. Optional: true T-streaming for 60k+ frame recordings

The flow above materialises the corrected movie in RAM (`movie` is a
float32 numpy array of shape `(T, H, W)`). On a 10k × 600 × 600 movie
that is ~14 GB — fine on most workstations. For longer recordings
(60k+ frames, ~86 GB) the working set exceeds typical RAM.

The streaming alternative uses a **pixel-major** zarr alongside the
time-major `mc.zarr`. Pipeline:

1. `transpose_zarr_to_pixel_major(mc_zarr, mc_pixel_zarr)` — one-time
   disk pass that rewrites the corrected movie with chunks like
   `(4096 pixels, 2000 frames)`. Pixel-row reads (the access pattern
   extraction needs) become `O(B·T)` IO instead of `O(H·W·T)`.
2. `model.fit(mc_zarr, do_motion_correction=False, Y_flat_zarr=mc_pixel_zarr)`
   — extraction runs against the on-disk pixel-major store. The 3D
   `mc_zarr` is read only for the strided greedy-init sample
   (~`T / init_stride` frames). Peak RAM is `K·T·4` (traces) plus
   small per-batch buffers — independent of `T`.

Re-run this notebook end-to-end with the cells below replacing the
"Load the corrected movie" and "Run extraction" cells when your
recording is too large for the in-RAM flow.

In [ ]:
# --- Drop-in replacement for cells 4 + 12 when extracting from 60k+ frames.

# Transpose the corrected movie to a pixel-major layout (one-time disk pass).
# Re-running is idempotent: if mc_pixel.zarr already exists it's reused.
from cnmfe.io import (
    open_zarr,
    open_zarr_pixel_major,
    transpose_zarr_to_pixel_major,
)

MC_PIXEL_ZARR = MC_ZARR.parent / 'mc_pixel.zarr'

t0 = time.time()
transpose_zarr_to_pixel_major(
    MC_ZARR, MC_PIXEL_ZARR,
    pixel_chunk=4096,
    time_chunk=2000,
    src_batch_frames=2000,        # ~2000 frames per IO batch on the source
    skip_if_exists=True,
)
print(f'transpose elapsed: {time.time() - t0:.1f}s\n')

# Open both layouts.
mc_3d = open_zarr(MC_ZARR)                       # time-major, used for init
Y_flat_zarr = open_zarr_pixel_major(MC_PIXEL_ZARR)
print(f'mc_3d        : {mc_3d.shape}  {mc_3d.dtype}')
print(f'Y_flat_zarr  : {Y_flat_zarr.shape}  {Y_flat_zarr.dtype}')

# Streaming extraction — no full Y_flat copy in RAM.
model = CNMFe(params)
t0 = time.time()
model.fit(mc_3d, do_motion_correction=False, Y_flat_zarr=Y_flat_zarr)
fit_elapsed = time.time() - t0

K = model.A.shape[1]
print(f'\nExtracted {K} components in {fit_elapsed:.1f}s (streaming).')
print(f'  Peak RAM was bounded by K·T·4 (traces) + per-batch buffers,')
print(f'  independent of T.')